# Download All Google Sheets Revisions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kduvekot/ubl-gc/blob/claude/google-sheets-history-cQ6AV/notebooks/download-all-revisions.ipynb)

Downloads every revision of the UBL 2.5 Google Sheets as ODS files,
one at a time, saving each to Google Drive immediately.

**Crash-resilient:** If the runtime disconnects, re-run and it picks up
where it left off (skips already-downloaded revisions).

**No repo clone needed** — this notebook only uses Google Auth + Drive API.

## What it does

For each revision (newest → oldest):
1. Download as ODS via Drive API v2 `revisions/{id}` → `exportLinks`
2. Gzip it → `rev-{id}.ods.gz`
3. Save to Google Drive folder
4. Record content hash (from `content.xml` inside ODS) in manifest

## Output

```
Drive: ubl-gc-revisions/
├── ubl25_library/
│   ├── rev-2005.ods.gz
│   ├── rev-2004.ods.gz
│   ├── ...
│   └── rev-1.ods.gz
├── ubl25_documents/
│   ├── rev-NNNN.ods.gz
│   └── ...
├── manifest-ubl25_library.json
└── manifest-ubl25_documents.json
```

## Step 0: Authentication

In [ ]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from google.auth.transport.requests import Request as AuthRequest

creds, project = google.auth.default(
    scopes=['https://www.googleapis.com/auth/drive']
)
creds.refresh(AuthRequest())
TOKEN = creds.token
print(f'Token: {TOKEN[:15]}...{TOKEN[-4:]}')

## Step 1: Mount Drive & Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_DIR = Path('/content/drive/MyDrive/ubl-gc-revisions')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Drive output: {DRIVE_DIR}')

## Step 2: Configuration

In [ ]:
import json, hashlib, gzip, time, zipfile, io
from urllib.request import Request, urlopen
from urllib.error import HTTPError
from collections import Counter

# Google Sheets file IDs
SHEETS = {
    'ubl25_library':   '18o1YqjHWUw0-s8mb3ja4i99obOUhs-4zpgso6RZrGaY',
    'ubl25_documents': '1024Th-Uj8cqliNEJc-3pDOR7DxAAW7gCG4e-pbtarsg',
}

ODS_MIME = 'application/x-vnd.oasis.opendocument.spreadsheet'
ODS_MIME_ALT = 'application/vnd.oasis.opendocument.spreadsheet'


def api_get(url, binary=False):
    """Authenticated GET with retry + exponential backoff."""
    headers = {'Authorization': f'Bearer {TOKEN}'}
    for attempt in range(4):
        try:
            req = Request(url, headers=headers)
            with urlopen(req, timeout=120) as resp:
                data = resp.read()
                return resp.status, data if binary else json.loads(data)
        except HTTPError as e:
            if e.code in (429, 500, 502, 503):
                wait = 2 ** (attempt + 1)
                print(f'  Retry ({e.code}), waiting {wait}s...')
                time.sleep(wait)
                continue
            return e.code, e.read().decode(errors='replace')
        except Exception as e:
            if attempt < 3:
                wait = 2 ** (attempt + 1)
                print(f'  Error: {e}, retrying in {wait}s...')
                time.sleep(wait)
                continue
            return 0, str(e)
    return 0, 'max retries exceeded'


def ods_content_hash(ods_bytes):
    """Extract content.xml from ODS (ZIP) and SHA256 it."""
    try:
        with zipfile.ZipFile(io.BytesIO(ods_bytes)) as zf:
            content = zf.read('content.xml')
            return hashlib.sha256(content).hexdigest()
    except Exception:
        return None


print('Configuration OK')

## Step 3: List All Revisions

Enumerates every revision of both sheets via Drive API v2.

In [ ]:
all_revisions = {}

for sheet_key, file_id in SHEETS.items():
    print(f'\n=== {sheet_key} ===')

    revisions = []
    page_token = None

    while True:
        url = (f'https://www.googleapis.com/drive/v2/files/{file_id}/revisions'
               f'?maxResults=1000')
        if page_token:
            url += f'&pageToken={page_token}'

        status, data = api_get(url)
        if status != 200:
            print(f'  ERROR: HTTP {status}')
            break

        items = data.get('items', [])
        revisions.extend(items)

        page_token = data.get('nextPageToken')
        if not page_token:
            break
        print(f'  ... {len(revisions)} so far')

    all_revisions[sheet_key] = revisions
    print(f'  Total: {len(revisions)} revisions')

    if revisions:
        print(f'  First: rev-{revisions[0]["id"]} '
              f'({revisions[0].get("modifiedDate", "?")})')
        print(f'  Last:  rev-{revisions[-1]["id"]} '
              f'({revisions[-1].get("modifiedDate", "?")})')

print(f'\nTotal: {sum(len(v) for v in all_revisions.values())} revisions')

## Step 4: Download Loop

**Set `SHEET_KEY` below** to choose which sheet to download.

Works backwards (newest → oldest). Each revision is:
1. Downloaded as ODS
2. Content-hashed (via `content.xml`)
3. Gzipped and saved to Drive as `rev-{id}.ods.gz`
4. Recorded in `manifest-{sheet}.json`

**Resume-safe:** skips revisions that already have a `.ods.gz` on Drive.

In [ ]:
# ============================================================
# CHOOSE WHICH SHEET TO DOWNLOAD
# Run once with 'ubl25_library', then again with 'ubl25_documents'
# ============================================================
SHEET_KEY = 'ubl25_library'
# SHEET_KEY = 'ubl25_documents'
# ============================================================

file_id = SHEETS[SHEET_KEY]
revisions = all_revisions[SHEET_KEY]

# Output directory on Drive
out_dir = DRIVE_DIR / SHEET_KEY
out_dir.mkdir(exist_ok=True)

# Load existing manifest for resume
manifest_path = DRIVE_DIR / f'manifest-{SHEET_KEY}.json'
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    done_ids = {str(r['id']) for r in manifest.get('revisions', [])}
    print(f'Resuming: {len(done_ids)} already in manifest')
else:
    manifest = {'sheet_id': file_id, 'sheet_key': SHEET_KEY, 'revisions': []}
    done_ids = set()

# Also check for files on Drive not yet in manifest (belt & suspenders)
existing_files = {f.stem.replace('.ods', ''): f
                  for f in out_dir.glob('rev-*.ods.gz')}
print(f'Files on Drive: {len(existing_files)}')

# Work backwards (newest first)
rev_list = list(reversed(revisions))
total = len(rev_list)
skipped = 0
downloaded = 0
errors = 0
seen_hashes = set(r.get('content_hash') for r in manifest.get('revisions', [])
                  if r.get('content_hash'))

print(f'\nProcessing {total} revisions of {SHEET_KEY} (newest → oldest)')
print(f'Output: {out_dir}/')
print()

for i, rev in enumerate(rev_list):
    rev_id = str(rev['id'])
    modified = rev.get('modifiedDate', '?')
    gz_name = f'rev-{rev_id}.ods.gz'
    gz_path = out_dir / gz_name

    # Skip if already done
    if rev_id in done_ids:
        skipped += 1
        continue

    # Skip if file exists on Drive already
    if gz_path.exists() and gz_path.stat().st_size > 0:
        skipped += 1
        done_ids.add(rev_id)
        continue

    pct = (i + 1) / total * 100
    print(f'[{i+1}/{total} {pct:.0f}%] rev-{rev_id} ({modified})', end=' ')

    # Step 1: Get exportLinks
    v2_url = (f'https://www.googleapis.com/drive/v2/files/{file_id}'
              f'/revisions/{rev_id}')
    status, data = api_get(v2_url)

    if status != 200 or not isinstance(data, dict):
        print(f'ERROR getting exportLinks: HTTP {status}')
        errors += 1
        time.sleep(1)
        continue

    export_links = data.get('exportLinks', {})
    ods_url = export_links.get(ODS_MIME) or export_links.get(ODS_MIME_ALT)

    if not ods_url:
        print(f'ERROR: No ODS export link')
        errors += 1
        time.sleep(1)
        continue

    time.sleep(0.3)

    # Step 2: Download ODS
    status, ods_data = api_get(ods_url, binary=True)

    if status != 200 or not isinstance(ods_data, bytes):
        print(f'ERROR downloading ODS: HTTP {status}')
        errors += 1
        time.sleep(1)
        continue

    # Step 3: Hash content.xml
    content_hash = ods_content_hash(ods_data)
    ods_hash = hashlib.sha256(ods_data).hexdigest()
    is_new = content_hash and content_hash not in seen_hashes
    if content_hash:
        seen_hashes.add(content_hash)

    # Step 4: Gzip and save to Drive
    gz_data = gzip.compress(ods_data, compresslevel=6)
    gz_path.write_bytes(gz_data)

    # Step 5: Record in manifest
    entry = {
        'id': rev_id,
        'modifiedDate': modified,
        'ods_size': len(ods_data),
        'gz_size': len(gz_data),
        'ods_hash': ods_hash,
        'content_hash': content_hash,
    }
    manifest['revisions'].append(entry)
    done_ids.add(rev_id)
    downloaded += 1

    tag = ' NEW' if is_new else ''
    print(f'{len(ods_data):,}b → {len(gz_data):,}b '
          f'content={content_hash[:12] if content_hash else "??"}...{tag}')

    # Save manifest every 25 revisions
    if downloaded % 25 == 0:
        manifest_path.write_text(json.dumps(manifest, indent=2))
        unique_so_far = len(seen_hashes)
        print(f'  --- manifest saved: {downloaded} new, '
              f'{skipped} skipped, {unique_so_far} unique states ---')

    time.sleep(0.5)

# Final manifest save
manifest['unique_states'] = len(seen_hashes)
manifest['total_revisions'] = total
manifest_path.write_text(json.dumps(manifest, indent=2))

print(f'\n{"="*50}')
print(f'DONE: {SHEET_KEY}')
print(f'  Downloaded:     {downloaded}')
print(f'  Skipped:        {skipped}')
print(f'  Errors:         {errors}')
print(f'  Unique states:  {len(seen_hashes)}')
print(f'  Manifest:       {manifest_path}')

## Step 5: Analyze Unique Content States

Groups revisions by their `content.xml` hash to find how many
unique spreadsheet states exist across all revisions.

In [ ]:
# Load manifest(s)
for sheet_key in SHEETS:
    mp = DRIVE_DIR / f'manifest-{sheet_key}.json'
    if not mp.exists():
        print(f'{sheet_key}: not yet downloaded')
        continue

    m = json.loads(mp.read_text())
    revs = m.get('revisions', [])

    hash_counts = Counter(
        r['content_hash'] for r in revs if r.get('content_hash')
    )

    print(f'\n{"="*60}')
    print(f'{sheet_key}: {len(revs)} revisions, '
          f'{len(hash_counts)} unique content states')
    print(f'{"="*60}')

    # Show each unique state with its revision range
    print(f'\nUnique states (most common first):')
    for rank, (h, count) in enumerate(hash_counts.most_common(), 1):
        matching = sorted(
            [r for r in revs if r.get('content_hash') == h],
            key=lambda r: int(r['id'])
        )
        first = matching[0]
        last = matching[-1]
        print(f'  {rank:3d}. {h[:16]}... x{count:4d}  '
              f'rev-{first["id"]} to rev-{last["id"]}')

    # Check known revisions
    known = {
        'ubl25_library': {
            '1843': 'V1/V2', '1868': 'V3/V4',
            '1999': 'V5/V6', '2005': 'V7-V10',
        },
        'ubl25_documents': {
            '1793': 'V1/V2', '1803': 'V3', '1983': 'V4',
            '2190': 'V5-V7', '2200': 'V8', '2204': 'V9/V10',
        },
    }.get(sheet_key, {})

    if known:
        print(f'\nKnown CI-run revisions:')
        for rev_id, label in known.items():
            entry = next(
                (r for r in revs if str(r['id']) == rev_id), None
            )
            if entry:
                h = entry.get('content_hash', '?')
                count = hash_counts.get(h, 0)
                print(f'  rev-{rev_id} ({label}): '
                      f'{h[:16]}... ({count} revisions share this state)')
            else:
                print(f'  rev-{rev_id} ({label}): not downloaded yet')

## Next Steps

After downloading both sheets:

1. **Find unique states** — typically 100-300 unique content states
   out of 2,000+ revisions
2. **Build timeline** — pair library + documents revisions by timestamp
3. **Convert unique pairs** — only convert (library, documents) pairs
   that produce new .gc output
4. **Match CI runs** — compare .gc hashes against known CI workflow
   outputs to find which exact revision was live during each run

### Estimated data

| Item | Size |
|------|------|
| ODS files (all revisions, gzipped) | ~1.2 GB |
| Unique content states | ~100-300 |
| .gc output (unique states only) | ~74 MB gzipped |